# B2B Radar — HDBSCAN min-cluster-size grid

This notebook compares `[50, 100, 150, 250]` on the same persisted clustering UMAP matrix. Results are stored outside the immutable source run. Completed variants are checksum-verified and reused after a Colab disconnect.

In [ ]:
REPOSITORY_URL = "https://github.com/osmirnov34/b2b-radar.git"
CODE_REF = "main"  # Prefer a reviewed commit SHA.
PROJECT_DIR = "/content/b2b-radar-grid"
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/b2b-radar"
RUN_ID = "colab-full-001"
MIN_CLUSTER_SIZES = (50, 100, 150, 250)
MATCH_MINIMUM_OVERLAP_SHARE = 0.05
TOP_STABILITY_TRAJECTORIES = 30
RUN_GRID = False  # Set True only after checking RUN_ID and paths below.

In [ ]:
import platform
import subprocess
import sys
from pathlib import Path

if sys.version_info[:2] not in {(3, 11), (3, 12), (3, 13)}:
    raise RuntimeError(f"Unsupported Python {platform.python_version()}; expected 3.11-3.13")
project_root = Path(PROJECT_DIR)
if project_root.exists():
    raise FileExistsError(f"Grid checkout already exists; restart the runtime: {project_root}")
subprocess.run(["git", "clone", "--filter=blob:none", REPOSITORY_URL, str(project_root)], check=True)
subprocess.run(["git", "-C", str(project_root), "checkout", CODE_REF], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{project_root}[analysis,visualization]"], check=True)
sys.path.insert(0, str(project_root))

In [ ]:
from google.colab import drive

from src.ml import (
    ClusteringGridConfig,
    ClusterMatchingConfig,
    ClusterStabilityConfig,
    analyze_grid_stability,
    load_analysis_artifacts,
    match_grid_clusters,
    run_clustering_grid,
)

drive.mount("/content/drive")
run_dir = Path(DRIVE_PROJECT_DIR) / "ml-runs" / RUN_ID
artifacts = load_analysis_artifacts(run_dir)
reduced_path = Path(artifacts.clustering.reduced_path)
reduction_manifest_path = run_dir / "08-reduction" / "clustering-manifest.json"
corpus_manifest_path = run_dir / "07-corpus" / "corpus-manifest.json"
grid_dir = Path(DRIVE_PROJECT_DIR) / "ml-experiments" / RUN_ID / "min-cluster-size-grid"
for label, path in {
    "reduced": reduced_path,
    "reduction_manifest": reduction_manifest_path,
    "corpus_manifest": corpus_manifest_path,
}.items():
    if not path.is_file():
        raise FileNotFoundError(f"{label} not found: {path}")
print({"source_run": str(run_dir), "grid_output": str(grid_dir), "sizes": MIN_CLUSTER_SIZES})

In [ ]:
def print_grid_progress(message: str) -> None:
    """Print one immediately visible, checkpoint-safe grid progress message."""
    print(message, flush=True)


grid_manifest = None
if RUN_GRID:
    grid_config = ClusteringGridConfig(
        min_cluster_sizes=MIN_CLUSTER_SIZES,
        base_config=artifacts.clustering.config,
    )
    grid_manifest = run_clustering_grid(
        reduced_path,
        reduction_manifest_path,
        corpus_manifest_path,
        grid_dir,
        config=grid_config,
        progress=print_grid_progress,
    )
else:
    print("Grid execution is disabled. Set RUN_GRID=True after verifying the source paths.")

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

if grid_manifest is None:
    raise RuntimeError("Run or resume the guarded grid cell first.")
comparison = pd.DataFrame([
    {
        "min_cluster_size": result.min_cluster_size,
        "clusters": result.metrics.clusters,
        "outliers": result.metrics.outliers,
        "outlier_share": result.metrics.outlier_share,
        "mean_probability": result.metrics.mean_probability,
        "low_confidence_share": result.metrics.low_confidence_share,
        "smallest_cluster": result.metrics.smallest_cluster,
        "median_cluster_size": result.metrics.median_cluster_size,
        "largest_cluster": result.metrics.largest_cluster,
        "dominant_cluster_share": result.metrics.dominant_cluster_share,
        "relative_validity": result.metrics.relative_validity,
        "dbcv": result.metrics.dbcv,
        "warnings": " | ".join(result.warnings),
    }
    for result in grid_manifest.results
])
display(comparison.style.format({
    "outlier_share": "{:.2%}",
    "mean_probability": "{:.4f}",
    "low_confidence_share": "{:.2%}",
    "dominant_cluster_share": "{:.2%}",
}))
figure = px.line(
    comparison,
    x="min_cluster_size",
    y=["outlier_share", "mean_probability", "dominant_cluster_share"],
    markers=True,
    title="HDBSCAN grid: quality and coverage trade-offs",
)
figure.show()
px.line(
    comparison,
    x="min_cluster_size",
    y="clusters",
    markers=True,
    title="Number of clusters by min_cluster_size",
).show()

In [ ]:
matching_manifest, cluster_transitions = match_grid_clusters(
    grid_dir / "clustering-grid-manifest.json",
    config=ClusterMatchingConfig(minimum_overlap_share=MATCH_MINIMUM_OVERLAP_SHARE),
)
transition_frame = pd.DataFrame([item.model_dump(mode="json") for item in cluster_transitions])
display(transition_frame.style.format({
    "jaccard": "{:.2%}",
    "source_retention": "{:.2%}",
    "target_composition": "{:.2%}",
}, na_rep="—"))
material = transition_frame[
    transition_frame["source_cluster_id"].notna()
    & transition_frame["target_cluster_id"].notna()
].copy()
if material.empty:
    print("No material cross-variant cluster overlaps were found.")
else:
    source_nodes = [
        f"{int(row.source_min_cluster_size)}:{int(row.source_cluster_id)}"
        for row in material.itertuples()
    ]
    target_nodes = [
        f"{int(row.target_min_cluster_size)}:{int(row.target_cluster_id)}"
        for row in material.itertuples()
    ]
    node_names = list(dict.fromkeys(source_nodes + target_nodes))
    node_index = {name: index for index, name in enumerate(node_names)}
    figure_transitions = go.Figure(go.Sankey(
        node={"label": node_names, "pad": 12, "thickness": 14},
        link={
            "source": [node_index[name] for name in source_nodes],
            "target": [node_index[name] for name in target_nodes],
            "value": material["overlap_records"].tolist(),
            "color": [
                "rgba(31,119,180,0.65)" if primary else "rgba(160,160,160,0.25)"
                for primary in material["primary_match"]
            ],
        },
    ))
    figure_transitions.update_layout(
        title="Material cluster transitions (min_cluster_size:cluster_id)",
        height=800,
    )
    figure_transitions.show()

In [ ]:
stability_manifest, cluster_stability = analyze_grid_stability(
    grid_dir / "cluster-matching-manifest.json",
    config=ClusterStabilityConfig(),
)
stability_frame = pd.DataFrame([
    {
        **trajectory.model_dump(mode="json", exclude={"nodes"}),
        "path": " → ".join(
            f"{node.min_cluster_size}:{node.cluster_id}" for node in trajectory.nodes
        ),
        "maximum_records": max(node.records for node in trajectory.nodes),
    }
    for trajectory in cluster_stability
]).sort_values(
    ["grid_coverage", "minimum_jaccard", "maximum_records"],
    ascending=[False, False, False],
    na_position="last",
)
if "ambiguous_transition" not in stability_frame.columns:
    raise RuntimeError("Stability output is missing ambiguity provenance.")
display(stability_frame.style.format({
    "grid_coverage": "{:.2%}",
    "mean_jaccard": "{:.2%}",
    "minimum_jaccard": "{:.2%}",
    "mean_source_retention": "{:.2%}",
    "minimum_source_retention": "{:.2%}",
}, na_rep="—"))
variant_levels = pd.DataFrame([
    {"min_cluster_size": item.min_cluster_size, **{level.value: count for level, count in item.levels.items()}}
    for item in stability_manifest.variants
]).melt(id_vars="min_cluster_size", var_name="level", value_name="clusters")
px.bar(
    variant_levels,
    x="min_cluster_size",
    y="clusters",
    color="level",
    barmode="stack",
    title="Cluster stability levels by grid variant",
).show()
largest_trajectory_ids = set(
    stability_frame.nlargest(TOP_STABILITY_TRAJECTORIES, "maximum_records")["trajectory_id"]
)
trajectory_points = pd.DataFrame([
    {
        "trajectory_id": str(trajectory.trajectory_id),
        "min_cluster_size": node.min_cluster_size,
        "records": node.records,
        "level": trajectory.level.value,
    }
    for trajectory in cluster_stability
    if trajectory.trajectory_id in largest_trajectory_ids
    for node in trajectory.nodes
])
if not trajectory_points.empty:
    px.line(
        trajectory_points,
        x="min_cluster_size",
        y="records",
        color="trajectory_id",
        line_group="trajectory_id",
        markers=True,
        hover_data=["level"],
        title="Largest mutual-primary cluster trajectories",
    ).show()

## Interpretation guardrails

Do not select a value from outlier share alone. Cluster IDs are local to each variant: use the transition table, Jaccard, retention, composition, and split/merge status instead of comparing IDs directly. A primary link is a deterministic mutual-best overlap, not proof that two topics have identical meaning. Compare topic coherence, stability, source concentration, and manual review on validation data in later plan points. This grid changes only HDBSCAN; UMAP vectors and the development corpus stay fixed. Grid labels are experimental and do not replace the source run automatically.